# Chapter 05 · Notebook 06: torch.no_grad explained

**Optional math reference:** [Mathematical view](00_math_intuition.ipynb), available for later. These programming lessons can be followed independently.

[Chapter guide](README.md) · [Return to the model class](05_custom_pytorch_model.ipynb#training) · [Course index](../README.md)

**Current learning pause:** study this notebook after Lesson 03 of Notebook 05. Once these ideas are clear, return to Lesson 04 there. This notebook is self-contained and can also serve as a later reference.

`torch.no_grad()` temporarily switches off recording new operations for PyTorch's usual backward calculation. Predictions and loss can still be calculated. It does not freeze parameter values or clear stored gradients.

In ordinary gradient-tracking mode, PyTorch records a calculation history when operations involve tensors requiring gradients. That history is called a **computation graph**. A later `loss.backward()` follows it to calculate parameter gradients. Forward calculation records the history; backward calculates the gradients. They are separate actions.

During evaluation, we want predictions and loss but do not need backward. Skipping this recording saves memory and avoids its extra work. When setting initial values or manually updating parameters, we also want those assignments to happen without becoming part of the training calculation history.

## Contents

1. [Same answer, different tracking](#tracking)
2. [Backward and already stored gradients](#backward)
3. [Why we use it when assigning initial values](#initialization)
4. [Why parameters can still change inside it](#manual-update)
5. [Separate training calculations from evaluation](#training-and-evaluation)
6. [Reference: related operations have different jobs](#reference)

Run one lesson at a time. Start with Setup in a fresh kernel. The scalar weight examples use input 2, target 8, and starting weight 1 so the arithmetic stays familiar.

## Setup

Import PyTorch. Gradient recording is normally enabled before entering a `no_grad` block. On leaving the block, the previous setting is restored.


In [ ]:
import torch
print("PyTorch version:", torch.__version__)
print("Gradient recording enabled:", torch.is_grad_enabled())


<a id="tracking"></a>
## Lesson 01: the prediction stays the same; tracking changes

**Mathematical view:** [Related explanation and worked example](00_math_intuition.ipynb#model-state).

Create a scalar weight with `requires_grad=True`. With weight 1 and input 2, prediction is 2, and the squared error against target 8 is 36.

Calculate that prediction and loss twice with the same values. First use ordinary gradient tracking, then use `with torch.no_grad():`.

`with` applies the temporary setting to the indented block. It does not skip the block: the arithmetic runs. `grad_fn` tells us whether a result has a recorded backward operation; here we inspect whether it is present, without needing its internal name.

The tracked result has `requires_grad=True` and history for backward. The result calculated inside `no_grad` has `requires_grad=False` and no such history. The original weight still has `requires_grad=True`, so future calculations outside this block can track gradients again.

**Before running:** will the prediction or loss numbers change just because we use `no_grad`? Will the original weight become permanently untrainable?

**My prediction:** _write here_


In [ ]:
# Only the weight needs gradients in this example, not the supplied input or target.
w = torch.tensor(1., requires_grad=True)
x = torch.tensor(2.)
y = torch.tensor(8.)

# Ordinary calculation: record the operations for a potential backward call.
tracked_prediction = w * x
tracked_loss = (tracked_prediction - y) ** 2

# Same arithmetic, without recording these operations for backward.
with torch.no_grad():
    untracked_prediction = w * x
    untracked_loss = (untracked_prediction - y) ** 2
    print("Recording enabled inside block:", torch.is_grad_enabled())

print("Tracked prediction and loss:", tracked_prediction.item(), tracked_loss.item())
print("No-grad prediction and loss:", untracked_prediction.item(), untracked_loss.item())
print("Tracked loss requires gradients:", tracked_loss.requires_grad)
print("No-grad loss requires gradients:", untracked_loss.requires_grad)
print("Tracked loss has backward history:", tracked_loss.grad_fn is not None)
print("No-grad loss has backward history:", untracked_loss.grad_fn is not None)
print("Weight still requires gradients:", w.requires_grad)
print("Recording enabled after block:", torch.is_grad_enabled())


**My observation and explanation:** _write here_

<a id="backward"></a>
## Lesson 02: recording, backward, and stored gradients are different

**Mathematical view:** [Related explanation and worked example](00_math_intuition.ipynb#model-state).

A forward calculation with tracking does not populate `w.grad` by itself. After backward, this example's weight gradient is `2 × (prediction - target) × input = 2 × (2 - 8) × 2 = -24`.

Evaluation inside `no_grad` does not clear that stored gradient. To clear it, use `w.grad = None` or an optimizer's `zero_grad()`. Their job is different from disabling recording.

A loss calculated entirely in `no_grad` here has no backward history, so calling its `backward()` would raise an error. Leaving the block does not retroactively attach history to that result. Make a fresh tracked prediction and loss when you want to train.

**Before running:** will `w.grad` change to zero or None after a no-grad evaluation?

**My prediction:** _write here_


In [ ]:
# Start a fresh experiment so stored gradients are initially absent.
w = torch.tensor(1., requires_grad=True)
x = torch.tensor(2.)
y = torch.tensor(8.)

prediction = w * x
loss = (prediction - y) ** 2
print("Gradient after forward, before backward:", w.grad)
loss.backward()  # Now calculate the gradient from the recorded history.
print("Gradient after backward:", w.grad.item())

with torch.no_grad():
    evaluation_loss = (w * x - y) ** 2
print("Gradient still stored after evaluation:", w.grad.item())
print("Evaluation loss requires gradients:", evaluation_loss.requires_grad)
# evaluation_loss.backward() would fail: this result has no backward history.

w.grad = None  # Clearing a stored gradient is a separate, explicit action.
print("Gradient after clearing:", w.grad)

# Recompute outside no_grad to obtain a fresh history for training.
fresh_training_loss = (w * x - y) ** 2
fresh_training_loss.backward()
print("Gradient from a fresh tracked calculation:", w.grad.item())


**My observation and explanation:** _write here_

<a id="initialization"></a>
## Lesson 03: assign initial values without recording the assignments

**Mathematical view:** [Related explanation and worked example](00_math_intuition.ipynb#model-state).

We often wrote `with torch.no_grad():` around `fill_()` when choosing known starting weights and bias. These assignments should change the parameter values without being treated as operations in our training history.

PyTorch restricts certain in-place changes to parameters requiring gradients in ordinary tracking mode. This is why we use `no_grad` for deliberate parameter initialization rather than treating initialization as part of a differentiable forward calculation.

After initialization, the parameters still require gradients. A subsequent forward pass outside the block can be tracked as usual. `fill_()` changes the existing tensor; we keep the same registered parameter.

**Before running:** what prediction should weight 3 and bias 2 produce for input 2? Should the weights still be eligible for training after initialization?

**My prediction:** _write here_


In [ ]:
model = torch.nn.Linear(1, 1)

# Deliberately change the initial parameter values without recording assignments.
with torch.no_grad():
    model.weight.fill_(3.)
    model.bias.fill_(2.)

example_x = torch.tensor([[2.]])
with torch.no_grad():
    evaluation_prediction = model(example_x)
tracked_prediction = model(example_x)  # Recording is enabled again outside the block.

print("Assigned weight and bias:", model.weight.item(), model.bias.item())
print("Prediction:", evaluation_prediction.item())
print("Weight still requires gradients:", model.weight.requires_grad)
print("Bias still requires gradients:", model.bias.requires_grad)
print("Evaluation prediction requires gradients:", evaluation_prediction.requires_grad)
print("Fresh tracked prediction requires gradients:", tracked_prediction.requires_grad)


**My observation and explanation:** _write here_

<a id="manual-update"></a>
## Lesson 04: no_grad does not prevent parameter changes

**Mathematical view:** [Related explanation and worked example](00_math_intuition.ipynb#model-state).

Remember our manual SGD update: `weight -= learning_rate * gradient`. Backward calculates the gradient first, with tracking enabled. The update itself then happens inside `no_grad` so the assignment is not recorded as part of a new gradient calculation.

Starting weight 1 has gradient -24 in this scalar example. At learning rate 0.05, the updated weight is `1 - 0.05 × (-24) = 2.2`. This is an example of a parameter changing inside `no_grad`.

The context manager does not perform the update itself. The explicit subtraction changes the weight. It also does not clear the old gradient; before another training step, clear gradients and recompute the forward pass and backward at the updated parameter values.

Standard plain SGD handles its update without recording it, so our normal `optimizer.step()` does not require an extra `no_grad` block around the entire training step. Its forward calculation and loss must remain tracked for backward.

**Before running:** will the weight change? Does the block make `w.requires_grad` permanently false?

**My prediction:** _write here_


In [ ]:
w = torch.tensor(1., requires_grad=True)
x = torch.tensor(2.)
y = torch.tensor(8.)
learning_rate = 0.05

loss = (w * x - y) ** 2
loss.backward()  # Calculate the gradient with tracking enabled.
print("Weight before update:", w.item())
print("Gradient before update:", w.grad.item())

with torch.no_grad():
    w -= learning_rate * w.grad  # This explicit assignment changes the parameter.
    loss_after_update = (w * x - y) ** 2

print("Weight after update:", w.item())
print("Loss after update:", loss_after_update.item())
print("Weight still requires gradients:", w.requires_grad)
print("Old gradient still stored:", w.grad.item())


**My observation and explanation:** _write here_

<a id="training-and-evaluation"></a>
## Lesson 05: keep the training calculation tracked; evaluate afterward

**Mathematical view:** [Related explanation and worked example](00_math_intuition.ipynb#model-state).

This complete step uses a one-input linear layer with weight 1 and bias 0, input 2, and target 8. There are now two trainable values: weight and bias.

The forward pass and loss calculation used for training happen outside `no_grad`, so backward can calculate gradients. After updating, evaluate fresh predictions and loss inside `no_grad`. In a loop, repeat the same separation every step.

If we placed the training forward pass and loss calculation inside `no_grad`, the resulting loss would not have the history needed for backward in this example. That is why we do not wrap the whole training loop in it.

**Before running:** which part needs recording, and which part can skip it?

**My answer:** _write here_


In [ ]:
model = torch.nn.Linear(1, 1)
with torch.no_grad():
    model.weight.fill_(1.)
    model.bias.fill_(0.)

example_x = torch.tensor([[2.]])
example_y = torch.tensor([[8.]])
loss_fn = torch.nn.MSELoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.05, momentum=0, weight_decay=0)

# Training: record the forward calculation so backward can calculate gradients.
optimizer.zero_grad(set_to_none=True)
training_prediction = model(example_x)
training_loss = loss_fn(training_prediction, example_y)
training_loss.backward()
optimizer.step()

# Evaluation: calculate fresh values after updating, with no recording for backward.
with torch.no_grad():
    evaluation_prediction = model(example_x)
    evaluation_loss = loss_fn(evaluation_prediction, example_y)

print("Training loss before update:", training_loss.item())
print("Updated weight and bias:", model.weight.item(), model.bias.item())
print("Evaluation prediction:", evaluation_prediction.item())
print("Evaluation loss after update:", evaluation_loss.item())
print("Training loss requires gradients:", training_loss.requires_grad)
print("Evaluation loss requires gradients:", evaluation_loss.requires_grad)


**My observation and explanation:** _write here_

<a id="reference"></a>
## Reference: related operations have different jobs

| Operation | Its job |
|---|---|
| `with torch.no_grad():` | Temporarily skip recording new operations for backward |
| `loss.backward()` | Calculate gradients using recorded history |
| `optimizer.zero_grad()` | Clear stored gradients on the optimizer's parameters |
| `optimizer.step()` | Apply an update using stored gradients and the optimizer's rules |
| `parameter.requires_grad` | Indicate whether gradients are required for that parameter in tracked calculations |
| `result.detach()` | Obtain a tensor detached from the original result's backward history; it does not change the parameter's trainability |
| `result.item()` | Extract a Python number for logging or inspection |

### A later distinction: model.eval()

`model.eval()` and `no_grad()` have different jobs. Evaluation mode changes behavior of layers such as Dropout or BatchNorm; it does not switch off gradient recording. Our current models contain only a Linear layer, so that layer's prediction calculation does not change between training and evaluation modes. We will study mode-dependent layers separately. `no_grad()` also does not set evaluation mode for you.

### Details to revisit later

- Exiting a no-grad block restores the previous gradient-recording setting. It does not change existing parameter flags or erase histories constructed before entering the block.
- Operations in the block still run. Parameter assignments can still change values, and other model state may change if the forward calculation changes it. Evaluation is unchanged in our examples because no update is requested and Linear has no running-statistics update.
- The ordinary reverse-mode recording discussed here has advanced exceptions: tensor factories can explicitly create new tensors or Parameters requiring gradients inside a no-grad block. This does not make subsequent ordinary operations inside that block tracked. We do not need that exception for the exercises above.

After reviewing these lessons, return to [Notebook 05, Lesson 04](05_custom_pytorch_model.ipynb#training) and apply the familiar training sequence to the custom model class.

## Documentation

- [PyTorch: no_grad](https://docs.pytorch.org/docs/stable/generated/torch.no_grad.html)
- [PyTorch: autograd mechanics, no-grad mode](https://docs.pytorch.org/docs/stable/notes/autograd.html#no-grad-mode)
- [PyTorch: autograd mechanics, evaluation mode](https://docs.pytorch.org/docs/stable/notes/autograd.html#evaluation-mode-nn-module-eval)
